<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:28px 32px;margin-bottom:8px;box-sizing:border-box;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#4C8DFF;margin-bottom:10px;">FRANCE DATA MARKET &middot; EXPLORATION</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:28px;color:#F2F3F5;letter-spacing:-0.02em;margin-bottom:8px;">State of the data job market in France</div><div style="font-family:'Inter',-apple-system,'Segoe UI',Roboto,sans-serif;font-size:14px;color:#9BA1AC;">Manual exploration notebook, run against the current warehouse.duckdb. Structure mirrors dashboard/requetes.sql: same grain, same definitions, so figures found here and in the generated report never diverge.</div></div>

In [1]:
import duckdb
import pandas as pd
import plotly.io as pio

import sys
sys.path.insert(0, '../dashboard')
from theme import BLUE, AMBER  # noqa: F401  -- registers the 'dark' Plotly template

con = duckdb.connect('../data/warehouse.duckdb', read_only=True)
pio.templates.default = 'dark'

<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">00 &middot; SCOPE</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Corpus size and top-line KPIs</div></div>

In [2]:
scope = con.execute('''
    select count(*) as offers,
           count(case when is_canonical_listing then 1 end) as listings
    from fct_job_offer
''').df()
scope

,offers,listings
0,1132,981


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">01 &middot; MARKET FLOW</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">What appears, what disappears</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Measured on fct_weekly_market_flow (actual presence per pull), never on the accumulated fct_weekly_market corpus -- see the README for why.</div></div>

In [3]:
flow = con.execute('''
    select week_start_date, weeks_since_previous, active_offer_count,
           new_offer_count, exit_count, exit_rate_pct
    from fct_weekly_market_flow
    order by week_start_date
''').df()
flow

,week_start_date,weeks_since_previous,active_offer_count,new_offer_count,exit_count,exit_rate_pct
0,2026-07-13,<NA>,552,552,<NA>,NaN
1,2026-08-24,6,497,408,463,83.9
2,2026-08-31,1,494,20,26,5.2


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">02 &middot; COMPENSATION</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Median advertised salary</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Filtered on annual_salary_plausible, grouped by canonical listing.</div></div>

In [4]:
salary_by_category = con.execute('''
    select employer_category, count(*) as n, median(salary_min) as median_salary
    from fct_job_offer
    where salary_period = 'annual' and annual_salary_plausible
      and is_canonical_listing
    group by employer_category
    order by median_salary desc
''').df()
salary_by_category

,employer_category,n,median_salary
0,INTERMEDIARY_RECLASSIFIED,3,65000.0
1,ANONYMOUS,24,45000.0
2,INTERMEDIARY,97,44000.0
3,DIRECT_EMPLOYER,134,42500.0


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">03 &middot; SALARY TRANSPARENCY</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Salary disclosure in job offers</div></div>

In [5]:
transparency = con.execute('''
    select employer_category,
           round(100.0 * count(distinct case when salary_mentioned then job_offer_id end)
                 / nullif(count(distinct job_offer_id), 0), 1) as rate_pct
    from fct_job_offer
    where is_canonical_listing
    group by employer_category
    order by rate_pct desc
''').df()
transparency

,employer_category,rate_pct
0,INTERMEDIARY,54.6
1,DIRECT_EMPLOYER,37.8
2,INTERMEDIARY_RECLASSIFIED,12.5
3,ANONYMOUS,9.1


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">04 &middot; GEOGRAPHY</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Top communes by offer count</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Counted by offer, not by listing -- a position opened in several communes represents an opportunity in each.</div></div>

In [6]:
geo = con.execute('''
    select c.commune_name, count(distinct o.job_offer_id) as offer_count
    from fct_job_offer o
    join dim_commune c on c.commune_key = o.commune_key
    where c.commune_name is not null and c.commune_name != 'UNRESOLVED'
    group by c.commune_name
    order by offer_count desc
    limit 10
''').df()
geo

,commune_name,offer_count
0,Paris,165
1,Lyon,52
2,Nantes,32
3,Nanterre,32
4,Courbevoie,31
5,Toulouse,29
6,Lille,25
7,Bordeaux,16
8,Boulogne-Billancourt,14
9,Levallois-Perret,14


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">05 &middot; TECHNOLOGIES</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Most requested technologies</div><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;line-height:1.6;color:#9BA1AC;margin-top:6px;">Empty if the local extraction dump isn't available (CI_WITHOUT_EXTRACTION).</div></div>

In [7]:
skills = con.execute('''
    select t.technology, count(distinct t.job_offer_id) as offer_count
    from fct_job_offer_technology t
    join fct_job_offer o using (job_offer_id)
    where o.is_canonical_listing
    group by t.technology
    order by offer_count desc
    limit 10
''').df()
skills

,technology,offer_count
0,Python,298
1,SQL,271
2,Power BI,191
3,AWS,66
4,Databricks,66
5,Azure,63
6,Snowflake,57
7,Tableau,56
8,Spark,53
9,Git,52


<div style="margin-top:32px;margin-bottom:4px;"><div style="font-family:'JetBrains Mono','SF Mono','Consolas',monospace;font-size:11px;letter-spacing:0.14em;text-transform:uppercase;color:#9BA1AC;">06 &middot; DOMAINS</div><div style="font-family:'Archivo Black','Arial Black',sans-serif;font-size:20px;color:#F2F3F5;letter-spacing:-0.02em;margin-top:4px;">Domains of intervention</div></div>

In [8]:
domains = con.execute('''
    select d.normalized_domain, count(distinct d.job_offer_id) as offer_count
    from fct_job_offer_domain d
    join fct_job_offer o using (job_offer_id)
    where o.is_canonical_listing
      and d.normalized_domain in (select distinct canonical_domain from mapping_domaines)
    group by d.normalized_domain
    order by offer_count desc
''').df()
domains

,normalized_domain,offer_count
0,Data Analysis,196
1,Data Governance,181
2,Business Intelligence,140
3,Machine Learning,112
4,Data Engineering,104
5,Data Science,102
6,Data Quality,93
7,Project Management,89
8,Data Architecture,64
9,Cloud computing,56


In [9]:
con.close()